# Taller 1 - Gestión de Datos
## Grupo: Juan Camilo Guerrero, Maria Camila Rincón, Luis Alberto Banda, Juan Pablo Marin y Gabriel Camilo Velasquez
## 1. Datos

**Objetivo del análisis:** preparar y cargar las tablas del taller de forma reproducible, para poder explorarlas en las secciones siguientes.

**Descripción del dataset:** se encuentra en la carpeta `Data` del workspace y consta de tres archivos:

| Archivo | Formato | Contenido |
|---|---|---|
| `countries.csv` | CSV | Tabla de países con su código, latitud, longitud y nombre. |
| `diabetes.csv` | CSV | Tabla de mediciones clínicas de pacientes y un indicador de diabetes (`Outcome`). |
| `Currencies.json` | JSON | Tabla de monedas por país, con sus códigos y nombres. |

### 1.1 Librerías
Importamos las librerías que usaremos durante todo el taller.

In [4]:
# Librerías del taller
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

### 1.2 Archivos disponibles
Listamos los archivos de `Data` con su tamaño. Si alguno pesa menos de 1 KB, probablemente es un puntero de Git LFS (no el dato real) y detenemos la ejecución.

In [5]:
# Ubicamos la carpeta Data (el notebook vive en work/, Data está un nivel arriba)
RUTAS_POSIBLES = [Path("../Data"), Path("Data")]
DATA_DIR = next((r for r in RUTAS_POSIBLES if r.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "No se encontró la carpeta 'Data'. Si usas Docker, recuerda que solo "
        "./work está montada en el contenedor: monta también ./Data."
    )

# Listamos cada archivo (csv y json) con su tamaño
archivos = sorted(list(DATA_DIR.glob("*.csv")) + list(DATA_DIR.glob("*.json")))
for archivo in archivos:
    print(f"{archivo.name:<20} {archivo.stat().st_size / 1024:>8.1f} KB")

# Un archivo < 1 KB es casi seguro un puntero de Git LFS
pequenos = [a.name for a in archivos if a.stat().st_size < 1024]
if pequenos:
    raise SystemExit(
        f"Detenido: {pequenos} pesan menos de 1 KB (puntero de Git LFS). "
        "Ejecuta 'git lfs pull' para descargar los datos reales."
    )

Currencies.json          44.2 KB
countries.csv             8.5 KB
diabetes.csv             23.3 KB


### 1.3 Carga de tablas
Cargamos `countries`, `Currencies` y `diabetes`. Convertimos a fecha las columnas cuyo nombre contenga `date` o `timestamp`, o termine en `_at`, e imprimimos el `shape` de cada tabla.

In [6]:
def es_columna_fecha(nombre):
    n = str(nombre).lower()
    return "date" in n or "timestamp" in n or n.endswith("_at")

tablas = {
    "countries": pd.read_csv(DATA_DIR / "countries.csv"),
    "Currencies": pd.read_json(DATA_DIR / "Currencies.json"),
    "diabetes": pd.read_csv(DATA_DIR / "diabetes.csv"),
}

for nombre, df in tablas.items():
    cols_fecha = [c for c in df.columns if es_columna_fecha(c)]
    for c in cols_fecha:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    print(f"{nombre:<11} shape={df.shape}  columnas de fecha convertidas: {cols_fecha or 'ninguna'}")

countries, Currencies, diabetes = tablas["countries"], tablas["Currencies"], tablas["diabetes"]

countries   shape=(245, 4)  columnas de fecha convertidas: ninguna
Currencies  shape=(254, 5)  columnas de fecha convertidas: ninguna
diabetes    shape=(768, 9)  columnas de fecha convertidas: ninguna


## 2. Perfilamiento

Resumimos cada atributo de las tablas cargadas: tipo de almacenamiento, tipo conceptual, nulos, valores distintos y estadísticos básicos.

> **Importante:** la columna `tipo_conceptual_sugerido` sale de reglas simples (dtype, número de valores distintos y nombre de la columna). Es solo una **sugerencia**: el analista debe validarla.

In [7]:
def tipo_conceptual(serie):
    """Sugiere el tipo conceptual de una columna con reglas simples (validar manualmente)."""
    n = serie.dropna()
    if pd.api.types.is_datetime64_any_dtype(serie):
        return "fecha"
    if isinstance(serie.dtype, pd.CategoricalDtype) and serie.dtype.ordered:
        return "ordinal"
    # Identificador: todos los valores distintos (no flotantes) o nombre típico de código/id
    nombre = str(serie.name).lower()
    es_id_por_nombre = nombre == "id" or nombre.endswith("_id") or "code" in nombre
    if not pd.api.types.is_float_dtype(serie) and len(n) > 0 and (n.nunique() == len(n) or es_id_por_nombre):
        return "identificador"
    if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
        return "nominal"
    if n.nunique() <= 2:
        return "nominal"  # variable binaria
    return "discreta" if pd.api.types.is_integer_dtype(serie) else "continua"


def perfil(df, tabla):
    """Devuelve una fila de perfilamiento por cada atributo de df."""
    filas = []
    for col in df.columns:
        s = df[col]
        es_num = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
        es_fecha = pd.api.types.is_datetime64_any_dtype(s)
        moda = s.mode(dropna=True)
        filas.append({
            "atributo": col,
            "tabla": tabla,
            "dtype": str(s.dtype),
            "tipo_conceptual_sugerido": tipo_conceptual(s),
            "nulos": int(s.isna().sum()),
            "%_nulos": round(s.isna().mean() * 100, 2),
            "distintos": int(s.nunique(dropna=True)),
            "media": s.mean() if es_num else np.nan,
            "desv_est": s.std() if es_num else np.nan,
            "moda": moda.iloc[0] if len(moda) else np.nan,
            "minimo": s.min() if (es_num or es_fecha) else np.nan,
            "maximo": s.max() if (es_num or es_fecha) else np.nan,
        })
    return pd.DataFrame(filas)


# Aplicamos la función a todas las tablas cargadas y concatenamos
perfiles = pd.concat([perfil(df, nombre) for nombre, df in tablas.items()], ignore_index=True)
perfiles = perfiles.sort_values(["%_nulos", "tabla"], ascending=[False, True], ignore_index=True)
perfiles

,atributo,tabla,dtype,tipo_conceptual_sugerido,nulos,%_nulos,distintos,media,desv_est,moda,minimo,maximo
0,country,countries,object,identificador,1,0.41,244,NaN,NaN,AD,NaN,NaN
1,latitude,countries,float64,continua,1,0.41,244,16.253109,27.031206,-75.250973,-75.250973,77.553604
2,longitude,countries,float64,continua,1,0.41,244,13.294814,73.976477,-177.156097,-177.156097,179.414413
3,Country,Currencies,object,identificador,0,0.00,254,NaN,NaN,ALand Islands,NaN,NaN
4,Country code (Alpha-2),Currencies,object,identificador,0,0.00,254,NaN,NaN,AC,NaN,NaN
5,Country code (Alpha-3),Currencies,object,identificador,0,0.00,253,NaN,NaN,,NaN,NaN
6,Currency,Currencies,object,nominal,0,0.00,155,NaN,NaN,Euro,NaN,NaN
7,Currency code,Currencies,object,identificador,0,0.00,155,NaN,NaN,EUR,NaN,NaN
8,name,countries,object,identificador,0,0.00,245,NaN,NaN,Afghanistan,NaN,NaN
9,Pregnancies,diabetes,int64,discreta,0,0.00,17,3.845052,3.369578,1.0,0.000000,17.000000


## 2.1 Ejercicio 1

El perfilamiento mostró un nulo en `country`, `latitude` y `longitude` de la tabla `countries`. Primero ubicamos las filas con nulos y luego las cruzamos con `Currencies` para identificar a qué países corresponden.

In [8]:
# Filas de countries con algún nulo en country, latitude o longitude
cols_revisar = ["country", "latitude", "longitude"]
filas_nulas = countries[countries[cols_revisar].isna().any(axis=1)].copy()
filas_nulas["columnas_nulas"] = filas_nulas[cols_revisar].apply(
    lambda f: ", ".join(f.index[f.isna()]), axis=1
)
filas_nulas

,country,latitude,longitude,name,columnas_nulas
156,NaN,-22.95764,18.49041,Namibia,country
226,UM,NaN,NaN,U.S. Minor Outlying Islands,"latitude, longitude"


### Cruce con `Currencies`
Llave principal: `countries.country` = `Currencies."Country code (Alpha-2)"`. Si el código es nulo, el cruce por código no es posible; en ese caso usamos como llave alternativa el nombre (`countries.name` = `Currencies.Country`).

In [9]:
alpha2 = "Country code (Alpha-2)"

# 1) Cruce por código cuando el código existe
por_codigo = (
    filas_nulas.dropna(subset=["country"])
    .merge(Currencies, left_on="country", right_on=alpha2, how="left")
    .assign(llave_usada="código (Alpha-2)")
)

# 2) Cruce por nombre cuando el código es nulo
por_nombre = (
    filas_nulas[filas_nulas["country"].isna()]
    .merge(Currencies, left_on="name", right_on="Country", how="left")
    .assign(llave_usada="nombre")
)

cruce_nulos = pd.concat([por_codigo, por_nombre], ignore_index=True)
cruce_nulos[["name", "columnas_nulas", "llave_usada", "Country", alpha2,
             "Country code (Alpha-3)", "Currency", "Currency code"]]

,name,columnas_nulas,llave_usada,Country,Country code (Alpha-2),Country code (Alpha-3),Currency,Currency code
0,U.S. Minor Outlying Islands,"latitude, longitude",código (Alpha-2),U.S. Outlying Islands,UM,UMI,US Dollar,USD
1,Namibia,country,nombre,Namibia,NA,NAM,Namibian Dollar,NAD


### Preguntas 4, 5 y 6: llave de unión, cruce y comparación con el nombre

**4. Llave identificada:** `countries.country` = `Currencies."Country code (Alpha-2)"`, es decir, el código ISO 3166-1 de dos letras. (`Currencies` también trae el código Alpha-3, pero `countries` no lo tiene, así que Alpha-2 es la llave común.)

**¿Por qué es mejor el código ISO que el nombre?**
- **Es un estándar:** un código identifica un solo país y es el mismo en todas las fuentes. El nombre depende de cómo cada fuente lo escribe.
- **El nombre varía** entre tablas: corchetes (`Congo [DRC]`), acentos y codificación (`Côte d'Ivoire`), nombres alternativos (`Myanmar [Burma]`, `Macedonia [FYROM]`), etc. Un cruce exacto por texto pierde esas filas.
- **Es corto y no tiene ambigüedad** de mayúsculas, espacios, idioma o signos.

**5. Unión:** `pd.merge(..., how="outer", indicator=True)` por la llave ISO. Reportamos cuántos registros quedaron en ambas tablas, solo en `countries` y solo en `Currencies`, y listamos los que no cruzaron. Al final comparamos con el cruce por nombre para respaldar la respuesta a la pregunta 4.

**6. Unión usando el nombre como llave:** repetimos el `merge` con `countries.name` = `Currencies.Country` y comparamos cuántos registros cruzan frente al cruce por código ISO. La diferencia son los registros que se pierden.

**Causa (dos ejemplos):** el mismo país está escrito distinto en cada tabla, por lo que la comparación exacta de texto falla aunque el código ISO sea el mismo.
- **Formato del nombre:** `Myanmar [Burma]` (countries) frente a `Myanmar` (Currencies), ambos con el código `MM`.
- **Codificación de caracteres:** `Côte d'Ivoire` (countries) frente a `C�te d�Ivoire` (Currencies), ambos con el código `CI`. El carácter acentuado se dañó en `Currencies`.

El código de la celda siguiente cuantifica la pérdida y lista los países afectados.

In [10]:
alpha2 = "Country code (Alpha-2)"
etiquetas = {"both": "en ambas tablas", "left_only": "solo en countries", "right_only": "solo en Currencies"}

# Pregunta 4: ¿la llave identifica de forma única a cada país?
print("country (countries) único:", countries["country"].dropna().is_unique)
print("Alpha-2 (Currencies) único:", Currencies[alpha2].is_unique)

# Punto 5: unión outer por código ISO
cruce = countries.merge(Currencies, left_on="country", right_on=alpha2, how="outer", indicator=True)
cruce["_merge"] = cruce["_merge"].astype(str).map(etiquetas)

print("\nRegistros por resultado del cruce (llave: código ISO):")
display(cruce["_merge"].value_counts().rename("registros").to_frame())

# Registros que no cruzaron
print("Registros que NO cruzaron:")
no_cruzan = cruce[cruce["_merge"] != "en ambas tablas"]
display(no_cruzan[["_merge", "country", "name", alpha2, "Country"]]
        .sort_values(["_merge", "name", "Country"], ignore_index=True))

# Respaldo de la pregunta 4: mismo cruce usando el nombre como llave
cruce_nombre = countries.merge(Currencies, left_on="name", right_on="Country", how="outer", indicator=True)
cruce_nombre["_merge"] = cruce_nombre["_merge"].astype(str).map(etiquetas)
comparacion = pd.concat(
    {"código ISO": cruce["_merge"].value_counts(), "nombre": cruce_nombre["_merge"].value_counts()}, axis=1
)
print("Comparación de llaves (número de registros):")
display(comparacion)

# Punto 6: registros que se pierden al usar el nombre como llave
cruzan_codigo = comparacion.loc["en ambas tablas", "código ISO"]
cruzan_nombre = comparacion.loc["en ambas tablas", "nombre"]
print(f"Cruzan por código ISO: {cruzan_codigo} | por nombre: {cruzan_nombre} | se pierden: {cruzan_codigo - cruzan_nombre}")

# Causa: países con el mismo código ISO pero nombre escrito distinto en cada tabla
mismo_codigo = countries.merge(Currencies, left_on="country", right_on=alpha2)
nombre_distinto = mismo_codigo[mismo_codigo["name"] != mismo_codigo["Country"]]
print(f"Países con el mismo código pero nombre distinto: {len(nombre_distinto)}")
display(nombre_distinto[["country", "name", "Country"]].reset_index(drop=True))


country (countries) único: True
Alpha-2 (Currencies) único: True

Registros por resultado del cruce (llave: código ISO):


,registros
_merge,
en ambas tablas,243
solo en Currencies,11
solo en countries,2


Registros que NO cruzaron:


,_merge,country,name,Country code (Alpha-2),Country
0,solo en Currencies,NaN,NaN,AX,ALand Islands
1,solo en Currencies,NaN,NaN,AC,Ascension Island
2,solo en Currencies,NaN,NaN,IC,Canary Islands
3,solo en Currencies,NaN,NaN,CW,Curacao
4,solo en Currencies,NaN,NaN,NA,Namibia
5,solo en Currencies,NaN,NaN,BL,Saint BarthLemy
6,solo en Currencies,NaN,NaN,MF,Saint Martin (French Part)
7,solo en Currencies,NaN,NaN,CS,Serbia and Montenegro
8,solo en Currencies,NaN,NaN,SX,Sint Maarten (Dutch Part)
9,solo en Currencies,NaN,NaN,SS,South Sudan


Comparación de llaves (número de registros):


,código ISO,nombre
_merge,,
en ambas tablas,243,229
solo en Currencies,11,25
solo en countries,2,16


Cruzan por código ISO: 243 | por nombre: 229 | se pierden: 14
Países con el mismo código pero nombre distinto: 15


,country,name,Country
0,CC,Cocos [Keeling] Islands,Clipperton Island
1,CD,Congo [DRC],Congo - Kinshasa
2,CG,Congo [Republic],Congo - Brazzaville
3,CI,Côte d'Ivoire,C�te d�Ivoire
4,FK,Falkland Islands [Islas Malvinas],Falkland Islands
5,IM,Isle of Man,Isle Of Man
6,MK,Macedonia [FYROM],Macedonia
7,MM,Myanmar [Burma],Myanmar
8,PN,Pitcairn Islands,Pitcairn
9,PS,Palestinian Territories,Palestine


### Puntos 7 y 8: guardado de la unión y reportes

**7. Guardado:** exportamos con `to_csv(index=False)` el resultado de la unión por código ISO (el `merge` *outer* del punto 5, que incluye la columna `_merge` con el origen de cada fila). El archivo queda junto al notebook, en `work/Countries_Currencies.csv`.

**8. Reportes:**
- **Tendencia central:** media, mediana, desviación estándar, mínimo, máximo y cuartiles (Q1 y Q3) de `latitude` y `longitude`, calculados sobre `countries`. La fila `n` indica cuántos valores no nulos se usaron.
- **ydata-profiling:** reporte HTML de la unión, guardado en `work/Countries_Currencies_profiling.html`. Ábrelo en el navegador.

In [11]:
# Punto 7: guardar la unión por código ISO (cruce de la pregunta 5) en CSV
ARCHIVO_CSV = Path("Countries_Currencies.csv")
cruce.to_csv(ARCHIVO_CSV, index=False)
print(f"Guardado: {ARCHIVO_CSV} ({ARCHIVO_CSV.stat().st_size / 1024:.1f} KB, {cruce.shape[0]} filas x {cruce.shape[1]} columnas)")

# Punto 8a: tendencia central y dispersión de latitud y longitud (tabla countries)
tendencia = (
    countries[["latitude", "longitude"]]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .rename(index={"count": "n (sin nulos)", "mean": "media", "std": "desv_est", "min": "mínimo",
                   "25%": "Q1 (25%)", "50%": "mediana (50%)", "75%": "Q3 (75%)", "max": "máximo"})
)
display(tendencia.round(4))

# Punto 8b: reporte de ydata-profiling en HTML (sobre la unión guardada)
from ydata_profiling import ProfileReport

ARCHIVO_HTML = Path("Countries_Currencies_profiling.html")
reporte = ProfileReport(cruce, title="Countries_Currencies", minimal=False)
reporte.to_file(ARCHIVO_HTML)
print(f"Reporte HTML generado: {ARCHIVO_HTML} ({ARCHIVO_HTML.stat().st_size / 1024:.0f} KB)")

Guardado: Countries_Currencies.csv (21.3 KB, 256 filas x 10 columnas)


,latitude,longitude
n (sin nulos),244.0000,244.0000
media,16.2531,13.2948
desv_est,27.0312,73.9765
mínimo,-75.2510,-177.1561
Q1 (25%),-0.3017,-38.0920
mediana (50%),16.8692,18.1821
Q3 (75%),38.9652,49.0467
máximo,77.5536,179.4144


/tmp/ipykernel_110/1616183094.py:16: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 10/10 [00:00<00:00, 530.21it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Reporte HTML generado: Countries_Currencies_profiling.html (3487 KB)
